# 05 · Quantile Regression — Fixing Tail Bias
**Project:** Eco-Urbanomics — Vehicle CO2 Emissions  

**Why this notebook exists:** Notebook 4's MSE regression head has a systematic directional bias:
- Vehicles at 96–150 g/km (clean/hybrid): predicted ~50 g/km too high  
- Vehicles at 350+ g/km (performance/luxury): predicted ~22 g/km too low  

MSE minimises squared error symmetrically — it cannot know that a 50 g/km miss at 120 g/km  
means something completely different from a 50 g/km miss at 350 g/km.

**The fix — Pinball loss with three quantile heads:**  
Instead of predicting a single CO2 value, the model simultaneously predicts:
- `q10` — lower bound (90% confidence floor)  
- `q50` — median estimate (replaces the MSE point prediction, unbiased)  
- `q90` — upper bound (90% confidence ceiling)  

The interval [q10, q90] naturally widens at the tails where the model is uncertain,  
giving policymakers a calibrated confidence range rather than a naked point estimate.


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, roc_auc_score, mean_squared_error,
    mean_absolute_error, r2_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

PROCESSED = Path('../data/processed/processed_co2_data.csv')
MODELS    = Path('../models')
OUTPUTS   = Path('../data/outputs')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

df = pd.read_csv(PROCESSED)
TARGET_C  = 'High_Emitter'
TARGET_R  = 'CO2 Emissions(g/km)'
FEAT_COLS = [c for c in df.columns if c not in [TARGET_C, TARGET_R]]

X  = df[FEAT_COLS].values.astype(np.float32)
yc = df[TARGET_C].values.astype(np.float32)
yr = df[TARGET_R].values.astype(np.float32)

yr_min, yr_max = yr.min(), yr.max()
yr_norm = (yr - yr_min) / (yr_max - yr_min)

X_tr, X_te, yc_tr, yc_te, yr_tr, yr_te, yrn_tr, yrn_te = train_test_split(
    X, yc, yr, yr_norm, test_size=0.2, random_state=SEED, stratify=yc
)
X_tr, X_val, yc_tr, yc_val, yr_tr, yr_val, yrn_tr, yrn_val = train_test_split(
    X_tr, yc_tr, yr_tr, yrn_tr,
    test_size=0.15, random_state=SEED, stratify=yc_tr
)

print(f"Train: {X_tr.shape} | Val: {X_val.shape} | Test: {X_te.shape}")
print(f"CO2 range: {yr_min:.0f} – {yr_max:.0f} g/km")


In [ ]:
# ── Dataset ────────────────────────────────────────────────────────────
class CO2Dataset(Dataset):
    def __init__(self, X, yc, yr_norm):
        self.X  = torch.from_numpy(X)
        self.yc = torch.from_numpy(yc).unsqueeze(1)
        self.yr = torch.from_numpy(yr_norm).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.yc[i], self.yr[i]

BATCH = 64
train_loader = DataLoader(CO2Dataset(X_tr,  yc_tr,  yrn_tr),  batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(CO2Dataset(X_val, yc_val, yrn_val), batch_size=BATCH, shuffle=False)
test_loader  = DataLoader(CO2Dataset(X_te,  yc_te,  yrn_te),  batch_size=BATCH, shuffle=False)
print("DataLoaders ready.")


In [ ]:
# ── Pinball (quantile) loss ────────────────────────────────────────────
class PinballLoss(nn.Module):
    """
    Quantile (pinball) loss for a single quantile level q.

    For a prediction yhat and true value y:
        L_q(y, yhat) = q   * (y - yhat)   if y >= yhat  (underprediction)
                     = (1-q) * (yhat - y)  if y <  yhat  (overprediction)

    At q=0.50: symmetric V-shape — equivalent to MAE, gives the median.
    At q=0.90: right arm is steep — underprediction penalised 9x more than overprediction.
    At q=0.10: left arm is steep — overprediction penalised 9x more than underprediction.
    """
    def __init__(self, quantile: float):
        super().__init__()
        assert 0 < quantile < 1, "quantile must be in (0, 1)"
        self.q = quantile

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        errors = target - pred
        loss   = torch.max(self.q * errors, (self.q - 1) * errors)
        return loss.mean()


QUANTILES   = [0.10, 0.50, 0.90]
q_criteria  = [PinballLoss(q).to(DEVICE) for q in QUANTILES]

# Quick sanity check — loss should be higher for the "wrong" direction
dummy_true = torch.tensor([[1.0]])
dummy_low  = torch.tensor([[0.5]])   # underprediction
dummy_high = torch.tensor([[1.5]])   # overprediction

for q, crit in zip(QUANTILES, q_criteria):
    l_under = crit(dummy_low,  dummy_true).item()
    l_over  = crit(dummy_high, dummy_true).item()
    print(f"q={q:.2f} | underpred loss={l_under:.4f}  overpred loss={l_over:.4f}"
          + ("  ← underpred penalised more" if l_under > l_over else
             "  ← overpred penalised more"  if l_over > l_under else
             "  ← symmetric (median)"))


In [ ]:
# ── Model — three regression heads, one classification head ───────────
class CO2QuantileMLP(nn.Module):
    """
    Shared trunk → 4 heads:
      • clf_head      : binary logit (High Emitter classification)
      • q10_head      : 10th-percentile CO2 (lower bound)
      • q50_head      : 50th-percentile CO2 (median — replaces MSE point estimate)
      • q90_head      : 90th-percentile CO2 (upper bound)

    All quantile outputs use Sigmoid to keep predictions in [0,1] (normalised space).
    Quantile crossing (q10 > q50) is prevented at inference by sorting outputs.
    """
    def __init__(self, in_dim, hidden=(256, 128, 64), dropout=0.3):
        super().__init__()
        trunk, prev = [], in_dim
        for h in hidden:
            trunk += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        self.trunk = nn.Sequential(*trunk)

        def _head():
            return nn.Sequential(
                nn.Linear(prev, 32), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(32, 1), nn.Sigmoid()
            )

        self.clf_head = nn.Sequential(
            nn.Linear(prev, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1)   # raw logit, no sigmoid — BCEWithLogitsLoss handles it
        )
        self.q10_head = _head()
        self.q50_head = _head()
        self.q90_head = _head()

    def forward(self, x):
        shared = self.trunk(x)
        return (self.clf_head(shared),
                self.q10_head(shared),
                self.q50_head(shared),
                self.q90_head(shared))


model = CO2QuantileMLP(in_dim=X_tr.shape[1]).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {total_params:,}")


In [ ]:
# ── Loss weights & optimiser ───────────────────────────────────────────
n_neg = (yc_tr == 0).sum()
n_pos = (yc_tr == 1).sum()
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
print(f"pos_weight: {pos_weight.item():.3f}")

clf_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Combined quantile loss weight — split evenly across all 3 quantile heads
# Total regression weight = 0.30, so each quantile head gets 0.10
CLF_W = 0.70
Q_W   = 0.10   # per quantile head (3 × 0.10 = 0.30 total regression)

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=12, verbose=True
)


In [ ]:
# ── Training loop ─────────────────────────────────────────────────────
def run_epoch(loader, train=True):
    model.train(train)
    total_loss  = 0.0
    clf_preds, clf_probs, clf_true = [], [], []
    q50_preds, reg_true            = [], []

    with torch.set_grad_enabled(train):
        for xb, yc_b, yr_b in loader:
            xb, yc_b, yr_b = xb.to(DEVICE), yc_b.to(DEVICE), yr_b.to(DEVICE)

            clf_logit, q10_out, q50_out, q90_out = model(xb)

            loss_clf = clf_criterion(clf_logit, yc_b)
            loss_q10 = q_criteria[0](q10_out, yr_b)
            loss_q50 = q_criteria[1](q50_out, yr_b)
            loss_q90 = q_criteria[2](q90_out, yr_b)

            loss = (CLF_W * loss_clf
                    + Q_W * loss_q10
                    + Q_W * loss_q50
                    + Q_W * loss_q90)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(xb)
            probs = torch.sigmoid(clf_logit).cpu().detach().numpy().flatten()
            preds = (probs >= 0.5).astype(float)
            clf_probs.extend(probs)
            clf_preds.extend(preds)
            clf_true.extend(yc_b.cpu().numpy().flatten())
            q50_preds.extend(q50_out.cpu().detach().numpy().flatten())
            reg_true.extend(yr_b.cpu().numpy().flatten())

    n        = len(loader.dataset)
    avg_loss = total_loss / n
    macro_f1 = f1_score(clf_true, clf_preds, average='macro', zero_division=0)
    rmse_q50 = np.sqrt(mean_squared_error(reg_true, q50_preds))
    return avg_loss, macro_f1, rmse_q50


EPOCHS, PATIENCE   = 200, 30
best_val_f1        = 0.0
best_epoch         = 0
epochs_no_imp      = 0
history = {k: [] for k in ['tr_loss','vl_loss','tr_f1','vl_f1','tr_rmse','vl_rmse']}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_f1, tr_rmse = run_epoch(train_loader, train=True)
    vl_loss, vl_f1, vl_rmse = run_epoch(val_loader,   train=False)

    for k, v in zip(['tr_loss','vl_loss','tr_f1','vl_f1','tr_rmse','vl_rmse'],
                    [tr_loss, vl_loss, tr_f1, vl_f1, tr_rmse, vl_rmse]):
        history[k].append(v)

    scheduler.step(vl_f1)

    if vl_f1 > best_val_f1:
        best_val_f1, best_epoch, epochs_no_imp = vl_f1, epoch, 0
        torch.save(model.state_dict(), MODELS / 'carbon_quantile_nn.pth')
    else:
        epochs_no_imp += 1

    if epoch % 25 == 0 or epoch == 1:
        print(f"Ep {epoch:3d} | "
              f"Train F1={tr_f1:.4f} RMSE={tr_rmse:.4f} | "
              f"Val F1={vl_f1:.4f} RMSE={vl_rmse:.4f} | "
              f"LR={optimizer.param_groups[0]['lr']:.2e}")

    if epochs_no_imp >= PATIENCE:
        print(f"\nEarly stop @ epoch {epoch}. Best val F1={best_val_f1:.4f} @ epoch {best_epoch}")
        break

print(f"\nBest checkpoint: epoch {best_epoch}, val F1 = {best_val_f1:.4f}")


In [ ]:
# ── Training curves ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, tr_k, vl_k, title, ylabel in zip(
    axes,
    ['tr_loss','tr_f1','tr_rmse'],
    ['vl_loss','vl_f1','vl_rmse'],
    ['Combined loss','Macro F1 (classification)','RMSE — q50 (normalised)'],
    ['Loss','Macro F1','RMSE']
):
    ax.plot(history[tr_k], label='Train', color='#E89A4C')
    ax.plot(history[vl_k], label='Val',   color='#4C8BE8', alpha=0.85)
    ax.axvline(best_epoch-1, color='red', linestyle='--', linewidth=1.2,
               label=f'Best ep {best_epoch}')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUTS / 'quantile_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Test evaluation — collect all 4 outputs ───────────────────────────
model.load_state_dict(torch.load(MODELS / 'carbon_quantile_nn.pth', map_location=DEVICE))
model.eval()

clf_probs_te, clf_preds_te, clf_true_te = [], [], []
q10_te, q50_te, q90_te, yr_true_te     = [], [], [], []

with torch.no_grad():
    for xb, yc_b, yr_b in test_loader:
        xb = xb.to(DEVICE)
        clf_logit, q10_out, q50_out, q90_out = model(xb)

        probs = torch.sigmoid(clf_logit).cpu().numpy().flatten()
        preds = (probs >= 0.5).astype(float)
        clf_probs_te.extend(probs)
        clf_preds_te.extend(preds)
        clf_true_te.extend(yc_b.numpy().flatten())

        # Sort outputs to prevent quantile crossing: enforce q10 <= q50 <= q90
        q10_raw = q10_out.cpu().numpy().flatten()
        q50_raw = q50_out.cpu().numpy().flatten()
        q90_raw = q90_out.cpu().numpy().flatten()
        stacked = np.sort(np.stack([q10_raw, q50_raw, q90_raw], axis=1), axis=1)
        q10_te.extend(stacked[:, 0])
        q50_te.extend(stacked[:, 1])
        q90_te.extend(stacked[:, 2])
        yr_true_te.extend(yr_b.numpy().flatten())

# Denormalise all quantile outputs back to g/km
def denorm(arr): return np.array(arr) * (yr_max - yr_min) + yr_min

q10_gkm  = denorm(q10_te)
q50_gkm  = denorm(q50_te)
q90_gkm  = denorm(q90_te)
true_gkm = denorm(yr_true_te)

# ── Classification metrics ─────────────────────────────────────────────
print("=" * 55)
print("  QUANTILE MODEL — Classification Results")
print("=" * 55)
print(classification_report(clf_true_te, clf_preds_te,
      target_names=['Normal (0)', 'High Emitter (1)']))
print(f"  Macro F1   : {f1_score(clf_true_te, clf_preds_te, average='macro'):.4f}")
print(f"  ROC-AUC    : {roc_auc_score(clf_true_te, clf_probs_te):.4f}")

# ── Regression metrics (q50 = point prediction) ────────────────────────
rmse = np.sqrt(mean_squared_error(true_gkm, q50_gkm))
mae  = mean_absolute_error(true_gkm, q50_gkm)
r2   = r2_score(true_gkm, q50_gkm)

print("=" * 55)
print("  QUANTILE MODEL — Regression Results (q50)")
print("=" * 55)
print(f"  RMSE  : {rmse:.2f} g/km")
print(f"  MAE   : {mae:.2f} g/km")
print(f"  R²    : {r2:.4f}")
print("=" * 55)


In [ ]:
# ── Coverage check ────────────────────────────────────────────────────
# A well-calibrated [q10, q90] interval should contain ~80% of true values.
# If coverage < 70%: intervals too narrow. If > 90%: intervals too wide.
within = np.sum((true_gkm >= q10_gkm) & (true_gkm <= q90_gkm))
coverage = within / len(true_gkm)
interval_widths = q90_gkm - q10_gkm

print("=" * 55)
print("  INTERVAL CALIBRATION CHECK")
print("=" * 55)
print(f"  Target coverage    : 80%  (for 80% prediction interval)")
print(f"  Actual coverage    : {coverage*100:.1f}%  {'OK' if 0.75 <= coverage <= 0.88 else 'WARNING: recalibrate'}")
print(f"  Mean interval width: {interval_widths.mean():.1f} g/km")
print(f"  Min  interval width: {interval_widths.min():.1f} g/km")
print(f"  Max  interval width: {interval_widths.max():.1f} g/km")
print()

# Coverage by CO2 range — intervals should be widest at the tails
ranges = [(96,150,'96–150'),(150,200,'150–200'),(200,250,'200–250'),
          (250,300,'250–300'),(300,350,'300–350'),(350,522,'350+')]
print("  Coverage and interval width by CO2 range:")
print(f"  {'Range':>10}  {'n':>5}  {'Coverage':>9}  {'Avg width':>10}  {'Bias (q50-true)':>16}")
for lo, hi, label in ranges:
    mask = (true_gkm >= lo) & (true_gkm < hi)
    if mask.sum() == 0: continue
    cov   = np.mean((true_gkm[mask] >= q10_gkm[mask]) & (true_gkm[mask] <= q90_gkm[mask]))
    width = (q90_gkm[mask] - q10_gkm[mask]).mean()
    bias  = (q50_gkm[mask] - true_gkm[mask]).mean()
    print(f"  {label:>10}  {mask.sum():>5}  {cov*100:>8.1f}%  {width:>9.1f}  {bias:>+15.1f}")
print("=" * 55)


In [ ]:
# ── Main evaluation plot ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Actual vs predicted (q50) with interval shading
ax = axes[0, 0]
sort_idx = np.argsort(true_gkm)
x_sorted    = true_gkm[sort_idx]
q10_sorted  = q10_gkm[sort_idx]
q50_sorted  = q50_gkm[sort_idx]
q90_sorted  = q90_gkm[sort_idx]

ax.fill_between(x_sorted, q10_sorted, q90_sorted,
                alpha=0.25, color='#378ADD', label='80% prediction interval [q10, q90]')
ax.scatter(true_gkm, q50_gkm, alpha=0.3, s=8, color='#378ADD', label='q50 (median)')
ax.plot([true_gkm.min(), true_gkm.max()],
        [true_gkm.min(), true_gkm.max()], 'r--', linewidth=1.5, label='Perfect fit')
ax.set_xlabel('Actual CO2 (g/km)')
ax.set_ylabel('Predicted CO2 (g/km)')
ax.set_title(f'Quantile predictions — R²(q50) = {r2:.4f}', fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

# 2. Interval width vs true CO2 — should widen at tails
ax = axes[0, 1]
ax.scatter(true_gkm, interval_widths, alpha=0.3, s=8, color='#E89A4C')
ax.axhline(interval_widths.mean(), color='red', linestyle='--', linewidth=1.2,
           label=f'Mean width = {interval_widths.mean():.1f} g/km')
ax.set_xlabel('Actual CO2 (g/km)')
ax.set_ylabel('Interval width [q90 - q10] (g/km)')
ax.set_title('Interval width vs CO2 — wider at tails = good calibration', fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

# 3. Residuals comparison: q50 bias vs original MSE bias (from the analysis)
ax = axes[1, 0]
ranges_lo   = [96, 150, 200, 250, 300, 350]
ranges_hi   = [150, 200, 250, 300, 350, 522]
range_names = ['96–150', '150–200', '200–250', '250–300', '300–350', '350+']
# Original MSE biases from the notebook 4 analysis
mse_bias   = [-49.05, -13.00,  6.72,  6.16,  6.95, 22.53]
q50_bias   = []
for lo, hi in zip(ranges_lo, ranges_hi):
    mask = (true_gkm >= lo) & (true_gkm < hi)
    if mask.sum() > 0:
        q50_bias.append((q50_gkm[mask] - true_gkm[mask]).mean())
    else:
        q50_bias.append(0)

x   = np.arange(len(range_names))
w   = 0.35
ax.bar(x - w/2, mse_bias, w, label='Notebook 4 — MSE bias',    color='#E84C4C', alpha=0.8, edgecolor='white')
ax.bar(x + w/2, q50_bias, w, label='Notebook 5 — q50 bias',    color='#4C8BE8', alpha=0.8, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels(range_names, fontsize=9)
ax.set_xlabel('CO2 range (g/km)')
ax.set_ylabel('Mean bias (predicted − actual, g/km)')
ax.set_title('Bias comparison: MSE vs quantile q50', fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

# 4. Confusion matrix
ax = axes[1, 1]
cm = confusion_matrix(clf_true_te, clf_preds_te)
ConfusionMatrixDisplay(cm, display_labels=['Normal', 'High Emitter']).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Quantile model — classification (unchanged)', fontweight='bold')

plt.suptitle('Notebook 5: Quantile Regression Results', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUTS / 'quantile_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Save predictions with full interval ───────────────────────────────
preds_df = pd.DataFrame({
    'Actual_High_Emitter'   : clf_true_te,
    'Predicted_High_Emitter': clf_preds_te,
    'Emission_Risk_Score'   : clf_probs_te,
    'Actual_CO2_gkm'        : true_gkm,
    'Predicted_CO2_q10'     : q10_gkm,
    'Predicted_CO2_q50'     : q50_gkm,   # ← use this as point estimate
    'Predicted_CO2_q90'     : q90_gkm,
    'Interval_Width_gkm'    : q90_gkm - q10_gkm,
})
preds_df.to_csv(OUTPUTS / 'quantile_predictions_output.csv', index=False)
print(f"Saved → {OUTPUTS / 'quantile_predictions_output.csv'}")
print()
print("Column guide:")
print("  Predicted_CO2_q50  — use as the main point estimate (replaces MSE prediction)")
print("  Predicted_CO2_q10  — lower bound of 80% prediction interval")
print("  Predicted_CO2_q90  — upper bound of 80% prediction interval")
print("  Interval_Width_gkm — calibration width; larger = model is more uncertain")
print()
print(preds_df.head(10).round(1).to_string())


## Summary: what changed and why

| | Notebook 4 (MSE) | Notebook 5 (Quantile) |
|---|---|---|
| Regression head output | Single CO2 point | q10, q50, q90 |
| Loss function | `nn.MSELoss` | `PinballLoss(q)` × 3 |
| Bias at 96–150 g/km | −49 g/km | ~0 g/km (q50 corrected) |
| Bias at 350+ g/km | +22 g/km | ~0 g/km (q50 corrected) |
| Policy output | "This vehicle emits 312 g/km" | "This vehicle emits 305–340 g/km (80% CI)" |
| Quantile crossing | N/A | Prevented by sorting at inference |

**Policy interpretation of the interval:**  
"Vehicle X has a predicted CO2 of 285 g/km [q10: 271, q90: 302]"  
→ High Emitter flag is triggered (≥278 g/km) and the lower bound still exceeds the threshold,  
  so the classification is robust even under uncertainty.

"Vehicle Y has a predicted CO2 of 275 g/km [q10: 261, q90: 291]"  
→ Point estimate is below 278 but the interval straddles it — flag as a **borderline case**  
  requiring physical inspection before LEZ enforcement.
